In [0]:
%run ./connectionNotebook

In [0]:
catalog_name = 'adbrag'
target_schema_name = 'gold'
src_schema_name = 'silver'

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, year, month, dayofweek

silver_df = spark.table(f"{catalog_name}.{src_schema_name}.orders_silver")

In [0]:
gold_df = (
    silver_df
    .withColumn(
        "order_amount_band",
        when(col("src_OrderAmount") < 100, "Small")
        .when((col("src_OrderAmount") >= 100) & (col("src_OrderAmount") < 500), "Medium")
        .otherwise("Large")
    )
    .withColumn(
        "order_day_type",
        when(dayofweek(col("src_OrderDate")).isin(1, 7), "Weekend").otherwise("Weekday")
    )
    .withColumn("order_year", year(col("src_OrderDate")))
    .withColumn("order_month", month(col("src_OrderDate")))
    .withColumn("gold_loaded_ts", current_timestamp())
)


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{target_schema_name}.orders_gold
(
    src_OrderID INT,
    src_CustomerID INT,
    src_OrderDate DATE,
    src_OrderAmount DECIMAL(10,2),
    processed_ts TIMESTAMP,
    order_amount_band STRING,
    order_day_type STRING,
    order_year INT,
    order_month INT,
    gold_loaded_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_OrderID)
""")



In [0]:
gold_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{target_schema_name}.orders_gold")